In [4]:
# Load libraries
import pandas as pd
import pyreadr
import numpy as np

In [ ]:
# 1. Load the R data file directly
print("Loading eth_edges.rds...")
result = pyreadr.read_r("eth_edges")
edges = result[None] # Extract the pandas dataframe

# Clean up addresses just in case
edges['from_address'] = edges['from_address'].str.lower()
edges['to_address'] = edges['to_address'].str.lower()

Loading eth_edges.rds...


In [5]:
print("1. Loading raw transaction data...")
# Load only the necessary columns to save memory
cols_to_use = ['hash', 'from_address', 'to_address', 'value', 'block_timestamp']
eth_data = pd.read_csv("eth_tx_last4days.csv", usecols=cols_to_use)

# Clean addresses
eth_data['from_address'] = eth_data['from_address'].str.lower()
eth_data['to_address'] = eth_data['to_address'].str.lower().fillna('__contract_creation__')
eth_data['value_eth'] = eth_data['value'] / 1e18

# Convert timestamp to datetime objects
eth_data['block_timestamp'] = pd.to_datetime(eth_data['block_timestamp'])

print("2. Calculating inter-transaction times (Burstiness)...")
# Sort chronologically to calculate the time between transactions
eth_data = eth_data.sort_values(by=['from_address', 'block_timestamp'])

# Calculate inter_tx_time (difference in seconds between consecutive transactions for each sender)
eth_data['inter_tx_time'] = eth_data.groupby('from_address')['block_timestamp'].diff().dt.total_seconds().fillna(0)


print("3. Building Node Features...")
# 'from' addresses (Outbound features)
from_nodes = eth_data.groupby('from_address').agg(
    n_tx_out=('hash', 'count'),
    mean_inter_tx_out=('inter_tx_time', 'mean'),
    activity_start_out=('block_timestamp', 'min'),
    activity_end_out=('block_timestamp', 'max'),
    total_eth_sent=('value_eth', 'sum')
).reset_index().rename(columns={'from_address': 'address'})

# 'to' addresses (Inbound features)
to_nodes = eth_data.groupby('to_address').agg(
    n_tx_in=('hash', 'count'),
    activity_start_in=('block_timestamp', 'min'),
    activity_end_in=('block_timestamp', 'max'),
    total_eth_received=('value_eth', 'sum')
).reset_index().rename(columns={'to_address': 'address'})

# Full list of unique nodes
all_addresses = pd.concat([eth_data['from_address'], eth_data['to_address']]).unique()
nodes = pd.DataFrame({'address': all_addresses})

# Merge them together
nodes = nodes.merge(from_nodes, on='address', how='left')
nodes = nodes.merge(to_nodes, on='address', how='left')

# Fill missing counts and values with 0
nodes[['n_tx_out', 'n_tx_in', 'total_eth_sent', 'total_eth_received']] = nodes[['n_tx_out', 'n_tx_in', 'total_eth_sent', 'total_eth_received']].fillna(0)

1. Loading raw transaction data...
2. Calculating inter-transaction times (Burstiness)...
3. Building Node Features...


In [ ]:
# Labeling nodes
# Load files
phish = pd.read_csv("phishing_addresses_detected.csv")
wash = pd.read_csv("wash_trading_addresses_detected.csv")
mixer = pd.read_csv("mixer_addresses_detected.csv")

# Clean the addresses (lowercase and strip whitespace)
phish['address'] = phish['address'].str.lower().str.strip()
wash['address'] = wash['address'].str.lower().str.strip()
mixer['address'] = mixer['address'].str.lower().str.strip()

# Combine all unique fraudulent addresses into one list
all_fraud_addresses = pd.concat([
    phish['address'], 
    wash['address'], 
    mixer['address']
]).unique()

print(f"-> Combined {len(all_fraud_addresses)} unique malicious addresses from your detections.")

# Assign binary label: 1 if it is in the fraud list, 0 if it is normal
nodes['is_fraud'] = nodes['address'].isin(all_fraud_addresses).astype(int)

print(f"-> Found {nodes['is_fraud'].sum()} known fraud nodes in our current network out of {len(nodes)} total nodes.")

-> Combined 1766 unique malicious addresses from your detections.
-> Found 1766 known fraud nodes in our current network out of 1829064 total nodes.


Neural networks cannot read Ethereum string addresses (like 0xabc123...). We have to map every unique address to an integer ID (from 0 to N-1).